In [4]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import numpy as np
from sklearn.model_selection import ParameterGrid

In [6]:
temp = pd.read_csv('.././data/development.csv', index_col=0)

remove = ['age', 'path', 'sampling_rate', 'gender', 'ethnicity']
temp['tempo'] = temp['tempo'].map(lambda x:x[1:-1]).astype(np.float32)

xtrain, xtest, ytrain, ytest = train_test_split(temp.drop(columns=remove), temp['age'], test_size=0.20)

lista = list()

for param in ParameterGrid({'hidden_layer_sizes': [(100,), (100, 100), (100, 100, 100), (100, 100, 100, 100)], 
                                         'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0], 
                                         'activation': ['identity', 'logistic', 'tanh', 'relu'],
                                         'max_iter':[500],
                                         'learning_rate': ['constant', 'invscaling', 'adaptive']}):
    lista.append([param, cross_val_score(MLPRegressor(**param), temp.drop(columns=remove), 
                                        temp['age'], cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1).mean()])
    

In [7]:
for param in sorted(lista, key=lambda x:x[1]):
    print(param[1], param[0])

-13.50361594425398 {'activation': 'identity', 'alpha': 0.0001, 'hidden_layer_sizes': (100, 100), 'learning_rate': 'invscaling', 'max_iter': 500}
-13.270950091860623 {'activation': 'identity', 'alpha': 0.001, 'hidden_layer_sizes': (100, 100), 'learning_rate': 'constant', 'max_iter': 500}
-13.091191282431756 {'activation': 'logistic', 'alpha': 0.0001, 'hidden_layer_sizes': (100, 100, 100), 'learning_rate': 'constant', 'max_iter': 500}
-13.091087878887496 {'activation': 'logistic', 'alpha': 0.001, 'hidden_layer_sizes': (100, 100, 100), 'learning_rate': 'constant', 'max_iter': 500}
-13.091024395724423 {'activation': 'logistic', 'alpha': 0.001, 'hidden_layer_sizes': (100, 100, 100), 'learning_rate': 'adaptive', 'max_iter': 500}
-13.091021942834033 {'activation': 'logistic', 'alpha': 0.0001, 'hidden_layer_sizes': (100, 100, 100, 100), 'learning_rate': 'adaptive', 'max_iter': 500}
-13.090817145553956 {'activation': 'logistic', 'alpha': 0.001, 'hidden_layer_sizes': (100, 100), 'learning_rate':

In [29]:
np.abs(cross_val_score(HistGradientBoostingRegressor(categorical_features=xtrain.dtypes=='object'), temp.drop(columns=remove), 
                temp['age'], cv=10, scoring='neg_root_mean_squared_error')).mean()

np.float64(10.856665413449619)

In [30]:
temp2 = temp.drop(remove, axis=1)
temp3 = pd.read_csv('.././data/evaluation.csv', index_col=0).drop(columns=['path'])

In [31]:
ypred = HistGradientBoostingRegressor(categorical_features=temp2.dtypes=='object').fit(temp2, temp['age']).predict(temp3)
ypred = pd.Series(ypred, name='Predicted')
pd.DataFrame(ypred, index=temp3.index).sort_index().to_csv('submission.csv') # hits 10.367 above naive baseline

In [45]:
temp = pd.read_csv('.././data/development.csv', index_col=0).drop(columns=['path'])
temp['tempo'] = temp['tempo'].map(lambda x:x[1:-1]).astype(np.float64)

In [50]:
temp['gender'] = temp['gender'].map(lambda x: 1 if x == 'female' else 0)

age = temp['age']
temp = temp.drop(columns=['age'])

In [51]:
clf = HistGradientBoostingRegressor(categorical_features=temp.dtypes=='object')
clf.fit(temp, age)

HistGradientBoostingRegressor(categorical_features=sampling_rate             False
gender                    False
ethnicity                  True
mean_pitch                False
max_pitch                 False
min_pitch                 False
jitter                    False
shimmer                   False
energy                    False
zcr_mean                  False
spectral_centroid_mean    False
tempo                     False
hnr                       False
num_words                 False
num_characters            False
num_pauses                False
silence_duration          False
dtype: bool)

In [59]:
from sklearn.inspection import permutation_importance
result = permutation_importance(clf, temp, age, n_repeats=10, random_state=42, n_jobs=-1)
idx = np.argsort(result['importances_mean'])[::-1]
for k,v in zip(temp.columns[idx], result['importances_mean'][idx]*100):
    print(k,v, sep=': ')

silence_duration: 42.777622183057986
ethnicity: 34.80645156190354
num_pauses: 18.09004719210531
spectral_centroid_mean: 15.378920772259299
zcr_mean: 14.958828084807454
jitter: 10.066153704718475
hnr: 7.780139203227726
shimmer: 7.143130951068552
mean_pitch: 6.876640284282668
max_pitch: 6.68491439772394
energy: 6.225300396175772
min_pitch: 5.643876337692592
tempo: 3.9945801300188855
num_characters: 0.06999814535945559
num_words: 0.015332458784832468
gender: 0.0
sampling_rate: 0.0


In [221]:
temp = pd.read_csv('.././data/development.csv', index_col=0).drop(columns=['path', 'sampling_rate'])
temp['tempo'] = temp['tempo'].map(lambda x:x[1:-1]).astype(np.float64)


In [222]:
etnicity = temp.copy()
topKNegri = set(etnicity['ethnicity'].value_counts().head(15).index)
topKNegri

{'arabic',
 'dutch',
 'english',
 'french',
 'german',
 'ibibio',
 'igbo',
 'italian',
 'japanese',
 'korean',
 'mandarin',
 'polish',
 'portuguese',
 'russian',
 'urhobo'}

In [223]:
etnie = etnicity['ethnicity'].map(lambda x: x if x in topKNegri else 'other')
genere = etnicity['gender'].map(lambda x: 0 if x == 'male' else 1)
etnicity = etnicity.drop(columns=['gender', 'ethnicity'])

In [224]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
age = etnicity['age']
etnicity = etnicity.drop(columns=['age'])

scaler.fit(etnicity)
etnicity = pd.DataFrame(scaler.transform(etnicity), columns=etnicity.columns, index=etnicity.index)

for val in topKNegri:
    etnicity[val] = etnie.map(lambda x: 1 if x == val else 0)

etnicity['gender'] = genere
etnicity['age'] = age

In [225]:
etnicity

,mean_pitch,max_pitch,min_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,...,korean,dutch,polish,igbo,russian,japanese,english,german,gender,age
Id,,,,,,,,,,,,,,,,,,,,,
0,1.526488,0.413221,-0.400362,-1.116336,-0.862609,-0.486481,2.039603,1.616870,0.796088,-1.287020,...,0,0,0,0,0,0,0,0,1,24.0
1,0.261473,0.411611,-0.415793,0.627830,-0.396497,0.429809,-0.502063,-0.843696,0.121675,-0.322635,...,0,0,0,0,0,0,0,0,1,22.5
2,0.346071,0.411505,-0.402148,-0.320350,0.403982,-0.367850,0.011442,0.691972,-0.225751,-0.622378,...,0,0,0,0,0,0,0,0,1,22.0
3,0.581502,0.410845,0.278367,-0.631887,-0.184536,2.825886,1.334833,1.888962,-0.225751,0.469989,...,0,0,0,0,0,0,1,0,1,22.0
4,1.205407,0.411146,-0.395821,1.032230,0.589336,0.026396,0.048513,-0.424086,-0.376805,-0.151479,...,0,1,0,0,0,0,0,0,0,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2928,1.090528,0.412179,-0.410228,0.371027,0.262767,-0.548247,0.136031,0.021567,1.759537,-0.686643,...,0,0,0,0,0,0,1,0,0,24.0
2929,-0.241320,0.384949,-0.359526,-0.886479,0.655138,-0.801732,-0.663606,0.925998,-1.234405,2.114540,...,0,0,0,1,0,0,0,0,1,15.0
2930,-0.471048,0.393440,0.542899,-1.737977,-0.145028,-0.616548,-0.897263,0.124957,-1.064363,0.536230,...,0,0,0,1,0,0,0,0,1,17.0


In [226]:
dfMan, dfWomen = etnicity[etnicity['gender']==0].drop(columns=['gender']), etnicity[etnicity['gender'] == 1].drop(columns=['gender'])

In [227]:
import matplotlib.pyplot as plt
import numpy as np

# Get the top 20 ethnicities from both datasets
ethnicity_men = dfMan['ethnicity'].value_counts().head(20)
ethnicity_women = dfWomen['ethnicity'].value_counts().head(20)

# Combine both datasets for a shared x-axis
combined_ethnicities = list(set(ethnicity_men.index).union(set(ethnicity_women.index)))

# Align the counts to the combined ethnicity list
men_counts = [ethnicity_men.get(ethnicity, 0) for ethnicity in combined_ethnicities]
women_counts = [ethnicity_women.get(ethnicity, 0) for ethnicity in combined_ethnicities]

# Set the positions for the bars
x = np.arange(len(combined_ethnicities))  # the label locations
width = 0.4  # the width of the bars

# Create the figure and axes
plt.figure(figsize=(12, 8))
plt.barh(x - width / 2, men_counts, width, label='Men', color='blue', alpha=0.7)
plt.barh(x + width / 2, women_counts, width, label='Women', color='orange', alpha=0.7)

# Add labels, legend, and title
plt.yticks(x, combined_ethnicities)  # Use combined ethnicities as y-axis labels
plt.xlabel('Count')
plt.title('Top 20 Ethnicities by Gender')
plt.legend()

# Show the plot
plt.tight_layout()
plt.show()


KeyError: 'ethnicity'

In [228]:
from sklearn.neural_network import MLPRegressor

clfMen = MLPRegressor()
clfWomen = MLPRegressor()
clfMen.fit(dfMan.drop(columns=['age']), dfMan['age'])
clfWomen.fit(dfWomen.drop(columns=['age']), dfWomen['age'])

c:\Users\utente\OneDrive\Desktop\Magistrale\01TWZSM_DataScienceLab\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\utente\OneDrive\Desktop\Magistrale\01TWZSM_DataScienceLab\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPRegressor()

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor, BaggingRegressor, StackingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor, Lars, LassoLars, OrthogonalMatchingPursuit, PassiveAggressiveRegressor, RANSACRegressor, SGDRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_score

models = [RandomForestRegressor(n_jobs=-1), GradientBoostingRegressor(), AdaBoostRegressor(), ExtraTreesRegressor(n_jobs=-1), BaggingRegressor(n_jobs=-1), 
          LinearRegression(), Ridge(), Lasso(), ElasticNet(), BayesianRidge(), HuberRegressor(), HistGradientBoostingRegressor(), StackingRegressor(n_jobs=-1),
          Lars(), LassoLars(), OrthogonalMatchingPursuit(), PassiveAggressiveRegressor(), RANSACRegressor(), SGDRegressor(), SVR(), KNeighborsRegressor(n_jobs=-1), DecisionTreeRegressor(), MLPRegressor()]

res = list()

for model in models:
    try:
        res.append([model.__str__(), np.abs(
            cross_val_score(model, dfMan.drop(columns=['age']), dfMan['age'], cv=10, scoring='neg_root_mean_squared_error').mean()+
            cross_val_score(model, dfWomen.drop(columns=['age']), dfWomen['age'], cv=10, scoring='neg_root_mean_squared_error').mean()
        )/2])
    except Exception:
        res.append([model.__str__(), 'error'])

c:\Users\utente\OneDrive\Desktop\Magistrale\01TWZSM_DataScienceLab\.venv\Lib\site-packages\sklearn\linear_model\_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
c:\Users\utente\OneDrive\Desktop\Magistrale\01TWZSM_DataScienceLab\.venv\Lib\site-packages\sklearn\linear_model\_huber.py:342: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)
c:\Users\utente\OneDrive\Desktop\Magistrale\01TWZSM_DataScienceLab\.venv\Lib\site-packages\sklearn\linear_model\_hub

In [220]:
for result in sorted(res, key=lambda x:x[1] if x[1] != 'error' else 1000):
    print(*result, sep='\t')

MLPRegressor()	10.556356862918136
GradientBoostingRegressor()	10.606930034801318
ExtraTreesRegressor(n_jobs=-1)	10.617178013458767
RandomForestRegressor(n_jobs=-1)	10.676164102843813
BayesianRidge()	10.682866184023922
SGDRegressor()	10.695944733163447
Ridge()	10.711719167906647
LinearRegression()	10.721643353878207
HistGradientBoostingRegressor()	10.77203079903504
Lars()	10.923328894890165
HuberRegressor()	11.028341058878656
LassoLars()	11.061611231930666
Lasso()	11.061625078424207
ElasticNet()	11.066886052713041
OrthogonalMatchingPursuit()	11.119787301960017
BaggingRegressor(n_jobs=-1)	11.13568411996945
SVR()	11.35992911785852
KNeighborsRegressor(n_jobs=-1)	11.533084588857932
AdaBoostRegressor()	12.1792335242083
PassiveAggressiveRegressor()	13.532105550623944
RANSACRegressor()	13.667641135634105
DecisionTreeRegressor()	15.037442062609838


In [230]:
predicted = pd.read_csv('.././data/evaluation.csv', index_col=0, header=0).drop(columns=['path', 'sampling_rate'])
predicted['tempo'] = predicted['tempo'].map(lambda x:x[1:-1]).astype(np.float64)

etnie = predicted['ethnicity'].map(lambda x: x if x in topKNegri else 'other')
genere = predicted['gender'].map(lambda x: 0 if x == 'male' else 1)
predicted = predicted.drop(columns=['gender', 'ethnicity'])


In [231]:
predicted = pd.DataFrame(scaler.transform(predicted), columns=predicted.columns, index=predicted.index)

for val in topKNegri:
    predicted[val] = etnie.map(lambda x: 1 if x == val else 0)

predicted['gender'] = genere

In [232]:
predictedMen, predictedWomen = predicted[predicted['gender']==0].drop(columns=['gender']), predicted[predicted['gender'] ==1].drop(columns=['gender'])

In [233]:
predictedSeries = pd.concat([pd.Series(clfMen.predict(predictedMen), index=predictedMen.index, name='Predicted'), 
                             pd.Series(clfWomen.predict(predictedWomen), index=predictedWomen.index, name='Predicted')], axis=0).sort_index()

In [234]:
predictedSeries.to_csv('submission.csv')

In [235]:
from sklearn.inspection import permutation_importance
result = permutation_importance(clfMen, dfMan.drop(columns=['age']), dfMan['age'], n_repeats=10, random_state=42, n_jobs=-1)
idx = np.argsort(result['importances_mean'])[::-1]
for k,v in zip(dfMan.drop(columns=['age']).columns[idx], result['importances_mean'][idx]*100):
    print(k,v, sep=': ')

silence_duration: 54.55323691409778
num_characters: 22.754886157838587
num_words: 21.37711569280433
spectral_centroid_mean: 16.877857199119028
max_pitch: 15.680209003307452
num_pauses: 12.053808262148705
mean_pitch: 10.486594949857677
zcr_mean: 8.234196534030273
hnr: 7.09157619737841
igbo: 6.688517886272102
english: 6.535802250243393
min_pitch: 4.712112951348478
jitter: 4.6884046460724065
shimmer: 1.4801478688720504
energy: 1.0886832774909294
arabic: 0.8800381591573847
tempo: 0.7757771677608061
italian: 0.676805685559484
mandarin: 0.3205608550334005
polish: 0.2007743767369985
ibibio: 0.18934967289875204
french: 0.13985102230580804
japanese: 0.11630839864309461
portuguese: 0.10951042503173247
dutch: 0.09635988227006287
korean: 0.05970622906166456
german: 0.04275489235836094
russian: 0.03860397556701867
urhobo: 0.018938007699750292


In [236]:
from sklearn.inspection import permutation_importance
result = permutation_importance(clfWomen, dfWomen.drop(columns=['age']), dfWomen['age'], n_repeats=10, random_state=42, n_jobs=-1)
idx = np.argsort(result['importances_mean'])[::-1]
for k,v in zip(dfWomen.drop(columns=['age']).columns[idx], result['importances_mean'][idx]*100):
    print(k,v, sep=': ')

silence_duration: 43.0992773686864
num_characters: 24.273699903219146
num_words: 23.039050965421946
num_pauses: 16.04771254777354
zcr_mean: 13.9231566748735
max_pitch: 13.213454786165238
spectral_centroid_mean: 11.300350888561539
min_pitch: 10.104475583350577
english: 6.528002284623288
jitter: 5.5104198473811445
hnr: 5.180022661224902
shimmer: 4.574821349973321
tempo: 1.819961245979852
igbo: 1.4374648175009097
mean_pitch: 1.0297417490796024
energy: 0.7365605425605237
arabic: 0.6410695776481856
italian: 0.5347958926543295
japanese: 0.41931432944235186
korean: 0.32772329047623505
mandarin: 0.28324043482496397
french: 0.2002014449045508
dutch: 0.16628580641126933
ibibio: 0.14094732460112946
polish: 0.13453115551834038
russian: 0.10445477045051056
german: 0.07253465272411885
portuguese: 0.07049217980960365
urhobo: 0.06550615994907139
